# Kaza — Phase 1 Training (Full WAXAL Live Streaming on A100)

Streams all 21 African languages directly from `google/WaxalNLP` (zero dataset storage, same-speaker & same-language packing, exact frame lengths `250 / 375 / 750`), trains Kaza with NoPE on the 6 new Global GQA layers, MTP (`t+2`, `t+4`), 8% slot corruption, and a 4-group AdamW (no-decay on norms/biases/embeddings + separate gentle ramp for the 6 new layers).

In [ ]:
import os
from google.colab import drive, userdata
drive.mount("/content/drive")

# Load HF_TOKEN from Colab Secrets (Key icon on left sidebar) to avoid HF Hub rate limits
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded from Colab Secrets.")
except Exception:
    print("Set HF_TOKEN in Colab Secrets (or os.environ['HF_TOKEN'] = 'hf_...') before streaming.")

!test -d /content/kaza && git -C /content/kaza pull --ff-only origin phase1-design || git clone -b phase1-design https://github.com/udohchuks/kaza.git /content/kaza
%cd /content/kaza
!pip install -q -r requirements.txt

In [ ]:
# Training & streaming configuration (fresh out_dir for 4-group AdamW + NoPE)
from pathlib import Path
from kaza.train import TrainConfig

DRIVE_ROOT = Path("/content/drive/MyDrive/kaza_phase1")
RUN_NAME = "waxal_nope_v1"

FRAME_LENGTHS = (250, 375, 750)  # 20s, 30s, 60s (exact divisors of 750; 1 length per step)
VAL_HOURS = 0.5                  # balanced across all ~21 WAXAL val splits (~0.5 MB)
DROP_PROB = 0.08                 # random slot zero-out on agent t-1
MTP_OFFSETS = (2, 4)             # training-only t+2 (160ms) & t+4 (320ms)
MTP_WEIGHT = 0.2
NOPE_NEW_LAYERS = True           # NoPE on the 6 new Global GQA layers

cfg = TrainConfig(
    out_dir=str(DRIVE_ROOT / "runs" / RUN_NAME),
    frame_lengths=FRAME_LENGTHS,
    val_seq_len=375,
    batch_size=4,            # A100-40GB: 4 (safe at 60s) | A100-80GB: 8 | L4: 2
    grad_accum=8,            # effective batch = 32 clips
    num_workers=2,
    max_steps=5000,
    warmup_freeze_steps=200,
    lr_warmup_steps=100,
    backbone_warmup_steps=100,
    lr=3e-4,                 # embeddings + depth decoder + MTP
    min_lr=3e-5,
    backbone_lr=1e-4,        # gentler LR for 6 new NoPE Global GQA layers
    min_backbone_lr=1e-5,
    log_every=10,
    eval_every=100,
    save_every=200,
    sample_every=200,
)
print(f"Run dir: {cfg.out_dir} | frame_lengths={cfg.frame_lengths} | eff_batch={cfg.batch_size * cfg.grad_accum}")

In [ ]:
# Discover all 21 WAXAL languages -> live same-speaker train stream + balanced val cache
from kaza.env import print_env
from kaza.codec import KazaCodec
from kaza.data import WaxalStreamingDataset, build_token_cache, discover_waxal_sources

env = print_env()
codec = KazaCodec(device=env.device, dtype=env.frozen_dtype)

train_sources, val_sources = discover_waxal_sources("google/WaxalNLP", include_tts=True)
train_stream = WaxalStreamingDataset(train_sources, clip_frames=max(FRAME_LENGTHS))

val_cache = build_token_cache(
    val_sources,
    codec=codec,
    out_path=DRIVE_ROOT / "data" / f"waxal_val_balanced_{VAL_HOURS}h.pt",
    max_hours=VAL_HOURS,
)
val_tokens = val_cache["tokens"]

In [ ]:
# Build KazaModel and run 1-step 60s preflight (verifies NoPE hook, peak VRAM, & throughput)
from kaza.model import KazaModel
from kaza.train import preflight_check

model = KazaModel(
    device=env.device,
    frozen_dtype=env.frozen_dtype,
    trainable_dtype=env.trainable_dtype,
    autocast_dtype=env.autocast_dtype,
    drop_prob=DROP_PROB,
    mtp_offsets=MTP_OFFSETS,
    mtp_weight=MTP_WEIGHT,
    nope_new_layers=NOPE_NEW_LAYERS,
)
n_train = sum(p.numel() for p in model.trainable_parameters()) / 1e6
print(f"Trainable params (incl. MTP): {n_train:.1f}M | NoPE new layers: {NOPE_NEW_LAYERS}")

preflight_check(model, codec, train_stream, cfg)

In [ ]:
# Run Phase 1 training (auto-resumes from latest.pt on Drive if interrupted)
from kaza.train import train_phase1

history = train_phase1(
    model,
    train_data=train_stream,
    val_tokens=val_tokens,
    cfg=cfg,
    codec=codec,
    resume=True,
)

In [ ]:
# Plot loss/accuracy curves and listen to best checkpoint vs held-out reference
import torch
from IPython.display import Audio as Play, display
from kaza.codec import SAMPLE_RATE
from kaza.train import plot_metrics, sample_continuation

plot_metrics(cfg.out_dir)

best_path = Path(cfg.out_dir) / "best.pt"
if best_path.exists():
    ckpt = torch.load(best_path, map_location="cpu", weights_only=False)
    model.load_trainable_state_dict(ckpt["model"])
    print(f"Loaded best.pt (step {ckpt['step']}, val_k12_loss={ckpt['best_val_loss']:.4f})")

ref_wav = Path(cfg.out_dir) / "samples" / "val_reference.wav"
if ref_wav.exists():
    print("Held-out reference (8s):")
    display(Play(str(ref_wav), rate=SAMPLE_RATE))

sample_continuation(model, codec, val_tokens, Path(cfg.out_dir), step=999999, temperature=0.8)